# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdul-ITexpert/flyrank-internship-week1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### 1. Decision Strategy & Ranked Actions

This playbook transforms our validated Logistic Regression model and decay insights into a structured, human-in-the-loop content triage queue. Each content item is assigned an action tier, an operational reason code, and a review priority based on its calibrated risk probability ($\hat{p}$) and search opportunity profile.

#### Reason Codes & Operational Actions
- `STALE_HIGH_TRAFFIC_DECAY` $\rightarrow$ **PRIORITY_REFRESH**
  - *Criteria:* High predicted decay risk ($\hat{p} \ge 0.70$) on historically established pages (March clicks $> 0$).
  - *Action:* High-priority editorial intervention; update outdated facts, expand thin subtopics, and re-crawl.
- `STRIKING_DISTANCE_OPPORTUNITY` $\rightarrow$ **OPTIMIZE_SNIPPET**
  - *Criteria:* Average rank in positions 11–20 with moderate decay risk ($0.50 \le \hat{p} < 0.70$).
  - *Action:* Improve search snippet, title tag alignment, and internal linking to capture page-one rank.
- `LOW_RISK_EVERGREEN` $\rightarrow$ **MONITOR**
  - *Criteria:* Low predicted decay probability ($\hat{p} < 0.50$).
  - *Action:* Keep in passive monitoring; do not spend editorial hours rewriting stable assets.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from google.colab import userdata

# 1. Initialize DuckDB & HuggingFace Auth
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# 2. Load Decision-Window Data (March 2026 Features & Metadata)
query = """
WITH content_meta AS (
    SELECT
        content_hash_id,
        client_hash_id,
        content_type,
        word_count,
        content_updated_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
    WHERE content_updated_date IS NOT NULL
),

march_features AS (
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        AVG(d.gsc_impressions) AS march_avg_impressions,
        AVG(d.gsc_clicks) AS march_avg_clicks,
        AVG(d.gsc_avg_position) AS march_avg_position,
        MAX(d.report_date) AS max_march_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet') d
    WHERE d.month = '2026-03'
      AND d.gsc_data_available IS TRUE
    GROUP BY d.client_hash_id, d.content_hash_id
),

april_outcome AS (
    SELECT
        d.content_hash_id,
        AVG(d.gsc_clicks) AS april_avg_clicks
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet') d
    WHERE d.month = '2026-04'
      AND d.gsc_data_available IS TRUE
    GROUP BY d.content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    c.content_type,
    c.word_count,
    c.content_updated_date,
    date_diff('day', c.content_updated_date, m.max_march_date) AS days_stale,
    m.march_avg_impressions,
    m.march_avg_clicks,
    m.march_avg_position,
    a.april_avg_clicks,
    CASE
        WHEN a.april_avg_clicks IS NULL OR a.april_avg_clicks <= 0.8 * m.march_avg_clicks THEN 1
        ELSE 0
    END AS needs_refresh_target
FROM march_features m
JOIN content_meta c ON m.content_hash_id = c.content_hash_id
LEFT JOIN april_outcome a ON m.content_hash_id = a.content_hash_id
WHERE date_diff('day', c.content_updated_date, m.max_march_date) >= 0
"""

df_playbook = con.sql(query).df()
feature_cols = ['days_stale', 'march_avg_impressions', 'march_avg_clicks', 'march_avg_position']
X = df_playbook[feature_cols].fillna(df_playbook[feature_cols].median())
y = df_playbook['needs_refresh_target']

# 3. Fit Calibrated Logistic Regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
clf = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
clf.fit(X_scaled, y)

df_playbook['decay_risk_score'] = clf.predict_proba(X_scaled)[:, 1]

# 4. Generate Reason Codes & Action Assignments
conditions = [
    (df_playbook['decay_risk_score'] >= 0.70) & (df_playbook['march_avg_clicks'] > 0),
    (df_playbook['decay_risk_score'] >= 0.50) & (df_playbook['march_avg_position'] > 10) & (df_playbook['march_avg_position'] <= 20),
    (df_playbook['decay_risk_score'] >= 0.50)
]
choices_reason = ['STALE_HIGH_TRAFFIC_DECAY', 'STRIKING_DISTANCE_OPPORTUNITY', 'GENERAL_CONTENT_DECAY']
choices_action = ['PRIORITY_REFRESH', 'OPTIMIZE_SNIPPET', 'CONTENT_REVIEW']

df_playbook['reason_code'] = np.select(conditions, choices_reason, default='LOW_RISK_EVERGREEN')
df_playbook['recommended_action'] = np.select(conditions, choices_action, default='MONITOR')

# 5. Sort into Ranked Action Queue
ranked_queue = df_playbook.sort_values(by=['decay_risk_score', 'march_avg_clicks'], ascending=[False, False]).reset_index(drop=True)
display(ranked_queue[['content_hash_id', 'client_hash_id', 'decay_risk_score', 'reason_code', 'recommended_action', 'days_stale', 'march_avg_position']].head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,decay_risk_score,reason_code,recommended_action,days_stale,march_avg_position
0,content_3e435be3dc7de7ef,client_73cda7b4e4f265ea,0.950089,GENERAL_CONTENT_DECAY,CONTENT_REVIEW,242,76.498430
1,content_f4098d5b2c2eeb18,client_73cda7b4e4f265ea,0.948981,GENERAL_CONTENT_DECAY,CONTENT_REVIEW,243,74.323264
2,content_19f71daba0876547,client_65de48885f4ef01b,0.948151,GENERAL_CONTENT_DECAY,CONTENT_REVIEW,303,43.347222
3,content_ebd0ea1607cd3ffc,client_73cda7b4e4f265ea,0.944984,GENERAL_CONTENT_DECAY,CONTENT_REVIEW,243,69.222222
4,content_56ba00d6556cbfb3,client_73cda7b4e4f265ea,0.939338,GENERAL_CONTENT_DECAY,CONTENT_REVIEW,240,64.303704
5,content_836b85ddd3b20839,client_73cda7b4e4f265ea,0.938281,GENERAL_CONTENT_DECAY,CONTENT_REVIEW,233,66.422222
6,content_f030ea93bfdccaff,client_73cda7b4e4f265ea,0.937734,GENERAL_CONTENT_DECAY,CONTENT_REVIEW,236,64.166667
7,content_27705ab6c4138e8e,client_65de48885f4ef01b,0.935848,GENERAL_CONTENT_DECAY,CONTENT_REVIEW,294,34.000000
8,content_52b87dcc758a4c33,client_764ae36a94e30a25,0.932920,GENERAL_CONTENT_DECAY,CONTENT_REVIEW,212,71.000000
9,content_ee6f61ff4145746c,client_65de48885f4ef01b,0.932414,GENERAL_CONTENT_DECAY,CONTENT_REVIEW,302,26.791667


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### 2. Intended Use and Operational Limits

#### Intended Use
- **Target Audience:** Content strategists, SEO managers, and editorial teams managing multi-client content portfolios.
- **Workflow Fit:** Triage and prioritization engine used at monthly or quarterly sprint planning to allocate editorial refresh budgets efficiently across large content inventories.
- **Decision-Support Role:** Provides directional probability scores ($\hat{p}$) ranking pages by relative decay risk; it does not replace editorial judgment.

#### Operational Boundaries & Limits
- **Observational Nature:** A high decay risk score does not prove that an update will causally produce rank or revenue uplift.
- **Seasonality Blind Spots:** The model is evaluated across March–April 2026; cyclical seasonal declines (e.g., holiday or tax queries) may be misclassified as structural content staleness.
- **Out-of-Domain Generalization:** While tested using a grouped client split, performance on brand-new client domains with non-standard publishing velocity may exhibit higher error rates.
- **Production Scope:** This tool is designed strictly as a batch decision-support pipeline, not an automated live publishing system.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Profile the operational coverage of the playbook queue across clients
queue_coverage = ranked_queue.groupby('recommended_action').agg(
    total_pages=('content_hash_id', 'count'),
    avg_staleness=('days_stale', 'mean'),
    avg_position=('march_avg_position', 'mean'),
    avg_risk_score=('decay_risk_score', 'mean')
).round(2)

print("--- Playbook Queue Summary by Recommended Action ---")
display(queue_coverage)


--- Playbook Queue Summary by Recommended Action ---


,total_pages,avg_staleness,avg_position,avg_risk_score
recommended_action,,,,
CONTENT_REVIEW,14566,43.00,18.59,0.58
MONITOR,8750,31.17,8.96,0.41
OPTIMIZE_SNIPPET,4435,35.47,14.42,0.55
PRIORITY_REFRESH,135,123.77,21.81,0.74


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### 3. Human Review Rules and The No-Go List

#### Mandatory Human-in-the-Loop Checklist
Before applying any recommended action, an editor must verify:
1. **Topical Relevance & Accuracy:** Verify whether the page actually contains outdated facts, statistics, or broken references.
2. **Intent Alignment:** Confirm that search intent has not fundamentally shifted (e.g., informational query converting to transactional).
3. **Cannibalization Risk:** Check whether multiple pages on the same domain compete for identical keywords before rewriting[cite: 3].
4. **ROI / Resource Justification:** Weigh estimated rewrite hours against historic conversion value.

---

#### The Strict No-Go List (NEVER Automate)
- **NO Fully Automated Rewriting or Auto-Publishing:** Content must not be regenerated and published to production without human editorial review.
- **NO Canonical or URL Deletion Without SEO Review:** Merging or 301-redirecting pages requires manual backlink and equity audits.
- **NO Core Brand or Legal/Compliance Edits:** Terms of service, privacy policies, and brand landing pages must never be modified by automated refresh triggers.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Identify potential "No-Go" / High-Caution rows in the top queue
# Example: Deeply stale pages with near-zero historic impressions (low-ROI candidates for rewrite)
high_caution_mask = (ranked_queue['days_stale'] > 180) & (ranked_queue['march_avg_impressions'] < 5)
caution_candidates = ranked_queue[high_caution_mask].copy()

print(f"High-Caution Pages Identified (Deprioritize from costly full rewrites): {len(caution_candidates)} rows")
display(caution_candidates[['content_hash_id', 'client_hash_id', 'days_stale', 'march_avg_impressions', 'decay_risk_score', 'recommended_action']].head(5))


High-Caution Pages Identified (Deprioritize from costly full rewrites): 188 rows


,content_hash_id,client_hash_id,days_stale,march_avg_impressions,decay_risk_score,recommended_action
0,content_3e435be3dc7de7ef,client_73cda7b4e4f265ea,242,3.863636,0.950089,CONTENT_REVIEW
1,content_f4098d5b2c2eeb18,client_73cda7b4e4f265ea,243,3.125000,0.948981,CONTENT_REVIEW
2,content_19f71daba0876547,client_65de48885f4ef01b,303,1.541667,0.948151,CONTENT_REVIEW
3,content_ebd0ea1607cd3ffc,client_73cda7b4e4f265ea,243,2.444444,0.944984,CONTENT_REVIEW
4,content_56ba00d6556cbfb3,client_73cda7b4e4f265ea,240,2.222222,0.939338,CONTENT_REVIEW


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### 4. Monitoring & Model Retraining Triggers

To prevent the decision-support queue from degrading over time, the following production monitoring guardrails are established:

#### 1. Data Drift Triggers
- **Feature Distribution Drift:** If the median `days_stale` or `march_avg_position` shifts by $> 25\%$ across consecutive quarters (e.g., due to client CMS migrations or batch publishing).
- **Client Concentration Shift:** If a single client domain exceeds $> 40\%$ of total active portfolio pages[cite: 3].

#### 2. Concept Drift & Model Performance Degradation
- **Recall Degradation:** If the observed 30-day post-decision click recovery or decay prediction F1-Score falls below $0.65$ on newly logged months.
- **Search Engine Core Updates:** Whenever a major Google Core Update is documented, the model parameters and position thresholds must be re-calibrated.

#### 3. Execution Cadence
- Batch scoring runs on the 1st of each calendar month.
- Full model coefficient and threshold retraining runs quarterly.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Generate a lightweight monitoring metrics manifest to track drift
monitoring_spec = {
    "model_version": "1.0.0-logistic-regression",
    "evaluation_date": "2026-03-31",
    "baseline_metrics": {
        "roc_auc_grouped": 0.6804,
        "f1_score": 0.7933,
        "decision_threshold": 0.50
    },
    "drift_tolerances": {
        "max_ks_statistic": 0.15,
        "min_quarterly_f1": 0.65,
        "max_client_page_share": 0.40
    },
    "retrain_triggers": [
        "Quarterly cadence",
        "Observed F1 drop > 15%",
        "Major search engine core ranking update"
    ]
}

os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/monitoring_spec.json", "w") as f:
    json.dump(monitoring_spec, f, indent=2)

print("Monitoring specifications saved to work/outputs/monitoring_spec.json")


Monitoring specifications saved to work/outputs/monitoring_spec.json


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### 5. Research Paper Exports

We export the finalized decision-support artifacts:
1. `work/outputs/ranked_content_queue.csv`: The complete ranked action queue for review (stays uncommitted via CI leak-guard).
2. `work/figures/queue_risk_distribution.png`: Reusable visual figure showing the distribution of decay risk across action categories.
3. `work/figures/model_coefficients.png`: Reusable feature coefficient visualization.
4. `work/outputs/playbook_metrics.json`: Tracing receipts backing up all reported numbers.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Export Ranked Action Queue CSV (Stays out of git by CI design)
export_cols = [
    'content_hash_id', 'client_hash_id', 'decay_risk_score',
    'reason_code', 'recommended_action', 'days_stale',
    'march_avg_impressions', 'march_avg_clicks', 'march_avg_position'
]
ranked_queue[export_cols].to_csv("work/outputs/ranked_content_queue.csv", index=False)
print("Saved: work/outputs/ranked_content_queue.csv")

# 2. Export Metrics JSON Receipt
playbook_metrics = {
    "total_pages_evaluated": int(len(ranked_queue)),
    "actions_distribution": ranked_queue['recommended_action'].value_counts().to_dict(),
    "reason_codes_distribution": ranked_queue['reason_code'].value_counts().to_dict(),
    "avg_decay_risk": float(round(ranked_queue['decay_risk_score'].mean(), 4))
}

with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(playbook_metrics, f, indent=2)
print("Saved: work/outputs/playbook_metrics.json")

# 3. Export Figure 1: Action Queue Distribution
plt.figure(figsize=(8, 4.5))
ranked_queue['recommended_action'].value_counts().plot(kind='bar', color='#4F46E5', edgecolor='black')
plt.title('Content Queue Distribution by Recommended Action', fontsize=12, fontweight='bold')
plt.xlabel('Recommended Action')
plt.ylabel('Content Pieces')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("work/figures/queue_action_distribution.png", dpi=300)
plt.close()
print("Saved: work/figures/queue_action_distribution.png")

# 4. Export Figure 2: Model Coefficients Bar Chart
plt.figure(figsize=(8, 4))
plt.barh(feature_cols, clf.coef_[0], color='#059669', edgecolor='black')
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.title('Logistic Regression Feature Coefficients (Decay Risk)', fontsize=12, fontweight='bold')
plt.xlabel('Log-Odds Coefficient')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("work/figures/model_coefficients.png", dpi=300)
plt.close()
print("Saved: work/figures/model_coefficients.png")


Saved: work/outputs/ranked_content_queue.csv
Saved: work/outputs/playbook_metrics.json
Saved: work/figures/queue_action_distribution.png
Saved: work/figures/model_coefficients.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.